In [ ]:
import torch
import sys
sys.path.append("D:/cxr-triage")

print("Path added")
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score

from src.data.dataset import ChestXrayDataset
from src.data.transforms import get_val_transforms
from src.models.densenet import DenseNetModel

ModuleNotFoundError: No module named 'src'

In [ ]:
LABELS = [
    'Atelectasis', 'Consolidation', 'Infiltration',
    'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis',
    'Effusion', 'Pneumonia', 'Pleural_Thickening',
    'Cardiomegaly', 'Nodule', 'Mass', 'Hernia'
]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

IMAGE_ROOT = "F:/X ray dataset/Second Version"

CHECKPOINT_PATH = (
    "D:/cxr-triage/checkpoints/clahe_320_logits_fix/best_model.pth"
)

TEST_CSV = "D:/cxr-triage/data/processed/test.csv"

In [ ]:
test_df = pd.read_csv(TEST_CSV)

test_dataset = ChestXrayDataset(
    csv_path=None,
    image_root=IMAGE_ROOT,
    transform=get_val_transforms()
)

test_dataset.df = test_df

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print(f"Test samples: {len(test_dataset)}")

In [ ]:
model = DenseNetModel(
    num_classes=14,
    pretrained=False
).to(DEVICE)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint['model_state_dict']
)

model.eval()
print("Model loaded successfully")

In [ ]:
all_targets = []
all_predictions = []

with torch.no_grad():
    for images, labels in test_loader:

        images = images.to(DEVICE)

        outputs = model(images)

        probs = torch.sigmoid(outputs)

        all_predictions.append(probs.cpu())
        all_targets.append(labels)

print("Inference complete")

In [ ]:
all_predictions = torch.cat(all_predictions, dim=0).numpy()
all_targets = torch.cat(all_targets, dim=0).numpy()

auc_scores = []

print("\nPer-Class AUC")
print("-" * 40)

for i, label in enumerate(LABELS):
    auc = roc_auc_score(
        all_targets[:, i],
        all_predictions[:, i]
    )

    auc_scores.append(auc)

    print(f"{label:<20} {auc:.4f}")

mean_auc = sum(auc_scores) / len(auc_scores)

print("-" * 40)
print(f"Mean AUC: {mean_auc:.4f}")